# Fabric Notebook: Parse Inforcer Assessment PDFs into Delta Tables

**Default lakehouse must be set to:** ManagedServiceData

**Reads PDFs from:**
- `Files/copilot_readiness/*.pdf` → copilot_readiness_assessments + _categories + _checks
- `Files/security_assessment/*.pdf` → security_assessment_assessments + _categories + _checks

**Tables are upserted (MERGE / delete+insert) so re-runs are idempotent.**

**Requires:** PyMuPDF (install via `%pip install pymupdf`)

In [1]:
%pip install pymupdf --quiet

StatementMeta(, f59b4128-12b6-46ac-bb19-2bb97b40a1df, 10, Finished, Available, Finished, True)


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [ ]:
# display(spark.table("copilot_readiness_assessments"))
# display(spark.table("copilot_readiness_categories"))
# display(spark.table("copilot_readiness_checks"))
# display(spark.table("security_assessment_assessments"))
# display(spark.table("security_assessment_categories"))
# display(spark.table("security_assessment_checks"))

In [8]:
# Verify the results - show table row counts
tables = [
    "copilot_readiness_assessments",
    "copilot_readiness_categories", 
    "copilot_readiness_checks",
    "security_assessment_assessments",
    "security_assessment_categories",
    "security_assessment_checks"
]

for table in tables:
    try:
        count = spark.table(table).count()
        print(f"✓ {table}: {count} rows")
    except Exception as e:
        print(f"✗ {table}: {str(e)}")

StatementMeta(, 3148d23d-e5b9-41c1-a07c-16c8301fc28c, 19, Finished, Available, Finished, False)

✗ copilot_readiness_assessments: No default context found, please attach a lakehouse before running spark sql queries with partial namespaces.
✗ copilot_readiness_categories: No default context found, please attach a lakehouse before running spark sql queries with partial namespaces.
✗ copilot_readiness_checks: No default context found, please attach a lakehouse before running spark sql queries with partial namespaces.
✗ security_assessment_assessments: No default context found, please attach a lakehouse before running spark sql queries with partial namespaces.
✗ security_assessment_categories: No default context found, please attach a lakehouse before running spark sql queries with partial namespaces.
✗ security_assessment_checks: No default context found, please attach a lakehouse before running spark sql queries with partial namespaces.


In [7]:
ingest_folder(COPILOT_FOLDER, "copilot_readiness")
ingest_folder(SECURITY_FOLDER, "security_assessment")

StatementMeta(, f59b4128-12b6-46ac-bb19-2bb97b40a1df, 17, Finished, Available, Finished, False)

Files/copilot_readiness: 1 PDF(s)
  parsed Assessment_CopilotReadiness_NAFDAC.pdf: 5 categories, 21 checks
  wrote 1 row(s) to copilot_readiness_assessments
  wrote 5 row(s) to copilot_readiness_categories


PySparkValueError: [CANNOT_DETERMINE_TYPE] Some of types cannot be determined after inferring.

In [18]:
# Debug: Check PDF file and extraction
test_files = list_pdfs(COPILOT_FOLDER)
if test_files:
    test_pdf = test_files[0]
    print(f"Testing PDF: {test_pdf.name}")
    print(f"Path: {test_pdf.path}")
    print(f"Size: {test_pdf.size} bytes")
    
    # Try reading raw bytes
    try:
        raw = notebookutils.fs.head(test_pdf.path, 100 * 1024 * 1024)
        print(f"Raw data type: {type(raw)}")
        print(f"Raw data length: {len(raw)}")
        
        # Convert to bytes if needed
        if isinstance(raw, str):
            raw_bytes = raw.encode("latin-1", errors="ignore")
        else:
            raw_bytes = raw
        
        print(f"Bytes length: {len(raw_bytes)}")
        print(f"First 100 bytes: {raw_bytes[:100]}")
        
        # Try opening with PyMuPDF
        doc = fitz.open(stream=raw_bytes, filetype="pdf")
        print(f"\nPDF opened successfully")
        print(f"Number of pages: {len(doc)}")
        
        if len(doc) > 0:
            page = doc[0]
            text = page.get_text()
            print(f"Page 1 text length: {len(text)}")
            print(f"Page 1 first 500 chars:\n{text[:500]}")
            
            # Try different extraction methods
            print(f"\n--- Trying 'blocks' method ---")
            blocks = page.get_text("blocks")
            print(f"Number of blocks: {len(blocks)}")
            if blocks:
                print(f"First block: {blocks[0]}")
        
        doc.close()
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

StatementMeta(, 726933ed-de73-42fd-8777-e6ba5267aa44, 16, Finished, Available, Finished, False)

Testing PDF: Assessment_CopilotReadiness_NAFDAC.pdf
Path: abfss://0f895a7e-09c6-4645-8b47-d272bc687b8a@onelake.dfs.fabric.microsoft.com/3d0144b0-12bf-4483-9508-67b26b1fd125/Files/copilot_readiness/Assessment_CopilotReadiness_NAFDAC.pdf
Size: 329547 bytes
Raw data type: <class 'str'>
Raw data length: 97202
Bytes length: 58490
First 100 bytes: b'%PDF-1.4\n%\n1 0 obj\n<</Title (Tenant Assessment: NAFDAC)\n/Creator (Mozilla/5.0 \\(X11; Linux x86_64\\) '

PDF opened successfully
Number of pages: 0


In [6]:
# --- Build rows + upsert ------------------------------------------------------

def build_rows(file_info, text, ingested_at):
    aid = assessment_id_for(file_info.path)
    header = parse_header(text)
    summary = parse_executive_summary(text)
    categories = parse_categories(text)
    sections = split_into_category_sections(text)

    assessment_row = Row(
        assessment_id=aid,
        file_name=file_info.name,
        file_path=file_info.path,
        assessment_name=header["assessment_name"],
        tenant_name=header["tenant_name"],
        assessment_date=header["assessment_date"],
        assessment_time=header["assessment_time"],
        overall_score_pct=summary["overall_score_pct"],
        passed_count=summary["passed"],
        failed_count=summary["failed"],
        warnings_count=summary["warnings"],
        ingested_at=ingested_at,
    )

    category_rows = [
        Row(
            assessment_id=aid,
            category_name=c["category_name"],
            score_pct=c["score_pct"],
            passed_count=c["passed"],
            failed_count=c["failed"],
            ingested_at=ingested_at,
        )
        for c in categories
    ]

    check_rows = []
    for category_label, area_code, block in sections:
        for c in parse_checks_in_block(block):
            check_rows.append(Row(
                assessment_id=aid,
                category_label=category_label,
                area_code=area_code,
                check_name=c["check_name"],
                status=c["status"],
                priority=c["priority"],
                framework_raw=c["framework_raw"],
                control=c["control"],
                level=c["level"],
                ingested_at=ingested_at,
            ))

    return assessment_row, category_rows, check_rows


def upsert(df, table_name, key_cols):
    if df.rdd.isEmpty():
        print(f"  no rows for {table_name}")
        return
    df.createOrReplaceTempView("staging")
    spark.sql(f"CREATE TABLE IF NOT EXISTS {table_name} USING DELTA AS SELECT * FROM staging WHERE 1=0")
    if len(key_cols) == 1:
        spark.sql(f"""
            MERGE INTO {table_name} t USING staging s ON t.{key_cols[0]} = s.{key_cols[0]}
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)
    else:
        # Child tables: delete prior rows for the affected assessment_ids, then append
        ids = [r.assessment_id for r in df.select("assessment_id").distinct().collect()]
        id_list = ",".join([f"'{i}'" for i in ids])
        spark.sql(f"DELETE FROM {table_name} WHERE assessment_id IN ({id_list})")
        df.write.mode("append").format("delta").saveAsTable(table_name)
    print(f"  wrote {df.count()} row(s) to {table_name}")


def ingest_folder(folder_path, table_prefix):
    files = list_pdfs(folder_path)
    print(f"{folder_path}: {len(files)} PDF(s)")
    if not files:
        return

    now = datetime.now(timezone.utc)
    all_assessments, all_categories, all_checks = [], [], []
    for f in files:
        try:
            text = read_pdf_text(f.path)
            a, cats, chks = build_rows(f, text, now)
            all_assessments.append(a)
            all_categories.extend(cats)
            all_checks.extend(chks)
            print(f"  parsed {f.name}: {len(cats)} categories, {len(chks)} checks")
        except Exception as e:
            print(f"  FAILED {f.name}: {e}")

    if all_assessments:
        upsert(spark.createDataFrame(all_assessments), f"{table_prefix}_assessments", ["assessment_id"])
    if all_categories:
        upsert(spark.createDataFrame(all_categories), f"{table_prefix}_categories", ["assessment_id", "category_name"])
    if all_checks:
        upsert(spark.createDataFrame(all_checks), f"{table_prefix}_checks", ["assessment_id", "check_name"])

StatementMeta(, f59b4128-12b6-46ac-bb19-2bb97b40a1df, 16, Finished, Available, Finished, False)

In [5]:
# --- Check-row parser ---------------------------------------------------------

CATEGORY_HEADER_RE = re.compile(
    r"([A-Z][A-Za-z0-9 &/]+?)\s*\(([^)]+)\)\s*\((\d+)\s*checks?\)"
)


def split_into_category_sections(text):
    start = text.find("Assessment Results by Category")
    if start == -1:
        return []
    body = text[start:]
    headers = list(CATEGORY_HEADER_RE.finditer(body))
    sections = []
    for i, h in enumerate(headers):
        end = headers[i + 1].start() if i + 1 < len(headers) else len(body)
        sections.append((h.group(1).strip(), h.group(2).strip(), body[h.end():end]))
    return sections


def _extract_control(framework_text):
    if not framework_text:
        return None
    m = re.search(r"Control:\s*([0-9.]+)", framework_text)
    return m.group(1) if m else None


def _extract_level(framework_text):
    if not framework_text:
        return None
    m = re.search(r"Level:\s*\(?(L[12])\)?", framework_text)
    return m.group(1) if m else None


def parse_checks_in_block(block):
    """
    Each check renders as:
        <check name lines>
        <sub-tag>
        <rationale paragraph...>
        <Status>           Failed | Passed | Warning
        <Priority>         High | Medium | Low
        <Framework lines>  e.g. "Copilot Readiness" OR
                                "CIS Microsoft 365 Foundations Benchmark v6.0.0"
                                "Control: 2.1.9"
                                "Level: (L1)"
    """
    lines = [ln.rstrip() for ln in block.split("\n")]
    checks = []
    n = len(lines)
    last_check_end = 0
    i = 0

    while i < n:
        line = lines[i].strip()
        if line in VALID_STATUSES:
            j = i + 1
            while j < n and not lines[j].strip():
                j += 1
            if j >= n:
                break
            priority = lines[j].strip()
            if priority not in VALID_PRIORITIES:
                i += 1
                continue

            # Collect framework lines after priority
            k = j + 1
            framework_lines = []
            while k < n:
                ln = lines[k].strip()
                if not ln:
                    p = k + 1
                    while p < n and not lines[p].strip():
                        p += 1
                    if p >= n:
                        break
                    nxt = lines[p].strip()
                    if nxt.startswith(("Control:", "Level:", "CIS ")) or nxt == "Copilot Readiness":
                        k = p
                        continue
                    break
                if ln in VALID_STATUSES:
                    break
                framework_lines.append(ln)
                k += 1

            framework_text = " | ".join(framework_lines).strip() or None

            # Check name = first few non-empty content lines between last_check_end and i
            name_lines = []
            for idx in range(last_check_end, i):
                t = lines[idx].strip()
                if not t:
                    if name_lines:
                        break
                    continue
                name_lines.append(t)
                if len(name_lines) >= 6:
                    break
            check_name = " ".join(name_lines).strip() or None

            checks.append({
                "check_name": check_name,
                "status": line,
                "priority": priority,
                "framework_raw": framework_text,
                "control": _extract_control(framework_text),
                "level": _extract_level(framework_text),
            })

            last_check_end = k
            i = k
            continue
        i += 1

    return checks

StatementMeta(, f59b4128-12b6-46ac-bb19-2bb97b40a1df, 15, Finished, Available, Finished, False)

In [4]:
# --- Shared parsers (both report types use the same Inforcer layout) ----------

def parse_header(text):
    out = {
        "assessment_name": None,
        "tenant_name": None,
        "assessment_date": None,
        "assessment_time": None,
    }
    m = re.search(r"Assessment:\s*\n([^\n]+)", text)
    if m:
        out["assessment_name"] = m.group(1).strip()
    m = re.search(r"Tenant Assessment:\s*\n([^\n]+)", text)
    if m:
        out["tenant_name"] = m.group(1).strip()
    m = re.search(r"Assessment Date:\s*\n([0-9\-/]+)", text)
    if m:
        out["assessment_date"] = m.group(1).strip()
    m = re.search(r"Assessment Time:\s*\n([0-9T:\-.Z]+)", text)
    if m:
        out["assessment_time"] = m.group(1).strip()
    return out


def parse_executive_summary(text):
    out = {"overall_score_pct": None, "passed": None, "failed": None, "warnings": None}
    m = re.search(r"Overall Score\s*\n\s*(\d+)\s*%", text)
    if m:
        out["overall_score_pct"] = int(m.group(1))
    m = re.search(r"Passed\s*\n\s*(\d+)\s*\n", text)
    if m:
        out["passed"] = int(m.group(1))
    m = re.search(r"Failed\s*\n\s*(\d+)\s*\n", text)
    if m:
        out["failed"] = int(m.group(1))
    m = re.search(r"Warnings\s*\n\s*(\d+)", text)
    if m:
        out["warnings"] = int(m.group(1))
    return out


def parse_categories(text):
    """
    "Assessment by Category (Top 5)" lines:
        M365 0%
        0 passed 2 failed
    """
    results = []
    pattern = re.compile(
        r"([A-Za-z0-9 &\-]+?)\s+(\d+)\s*%\s*\n\s*(\d+)\s+passed\s+(\d+)\s+failed",
        re.IGNORECASE,
    )
    for m in pattern.finditer(text):
        name = m.group(1).strip()
        if name.lower() in {"overall score"}:
            continue
        results.append({
            "category_name": name,
            "score_pct": int(m.group(2)),
            "passed": int(m.group(3)),
            "failed": int(m.group(4)),
        })
    return results

StatementMeta(, f59b4128-12b6-46ac-bb19-2bb97b40a1df, 14, Finished, Available, Finished, False)

In [3]:
# --- IO helpers ---------------------------------------------------------------

def list_pdfs(folder_path):
    try:
        entries = notebookutils.fs.ls(folder_path)
    except Exception as e:
        print(f"Folder {folder_path} not accessible: {e}")
        return []
    return [f for f in entries if f.name.lower().endswith(".pdf")]


def read_pdf_text(file_path):
    # Read binary PDF file using Spark
    pdf_df = spark.read.format("binaryFile").load(file_path)
    pdf_bytes = pdf_df.select("content").first()[0]
    
    # Open with PyMuPDF
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    text = "\n".join(page.get_text() for page in doc)
    doc.close()
    return text


def assessment_id_for(file_path):
    return hashlib.sha256(file_path.encode()).hexdigest()[:16]

StatementMeta(, f59b4128-12b6-46ac-bb19-2bb97b40a1df, 13, Finished, Available, Finished, False)

In [ ]:
import re
import hashlib
from datetime import datetime, timezone
import fitz  # PyMuPDF
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
import notebookutils

COPILOT_FOLDER = "Files/copilot_readiness"
SECURITY_FOLDER = "Files/security_assessment"

VALID_STATUSES = {"Passed", "Failed", "Warning"}
VALID_PRIORITIES = {"High", "Medium", "Low"}

# Define schemas for Delta tables
ASSESSMENT_SCHEMA = StructType([
    StructField("assessment_id", StringType(), False),
    StructField("file_name", StringType(), True),
    StructField("file_path", StringType(), True),
    StructField("assessment_name", StringType(), True),
    StructField("tenant_name", StringType(), True),
    StructField("assessment_date", StringType(), True),
    StructField("assessment_time", StringType(), True),
    StructField("overall_score_pct", IntegerType(), True),
    StructField("passed_count", IntegerType(), True),
    StructField("failed_count", IntegerType(), True),
    StructField("warnings_count", IntegerType(), True),
    StructField("ingested_at", TimestampType(), True),
])

CATEGORY_SCHEMA = StructType([
    StructField("assessment_id", StringType(), False),
    StructField("category_name", StringType(), True),
    StructField("score_pct", IntegerType(), True),
    StructField("passed_count", IntegerType(), True),
    StructField("failed_count", IntegerType(), True),
    StructField("ingested_at", TimestampType(), True),
])

CHECK_SCHEMA = StructType([
    StructField("assessment_id", StringType(), False),
    StructField("category_label", StringType(), True),
    StructField("area_code", StringType(), True),
    StructField("check_name", StringType(), True),
    StructField("status", StringType(), True),
    StructField("priority", StringType(), True),
    StructField("framework_raw", StringType(), True),
    StructField("control", StringType(), True),
    StructField("level", StringType(), True),
    StructField("ingested_at", TimestampType(), True),
])

StatementMeta(, f59b4128-12b6-46ac-bb19-2bb97b40a1df, 12, Finished, Available, Finished, False)